
# Tail results

This notebook computes the manuscript results from `rezultati.csv`.

The input CSV is expected to have columns `i`, `j`, `k`, `t`, `p`, `r`, and `data`, where `data` is a Python-style list in square brackets.

In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tail_recompute_lib as tr

DATA_PATH = Path("rezultati.csv")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

N_BOOT = 200

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)



## 1. data


In [2]:

raw = tr.read_raw(DATA_PATH)
print(f"Rows: {len(raw):,}")
print(f"Total numeric entries: {raw['n'].sum():,}")
print("Models:", sorted(raw['j'].unique()))
print("Prompts:", sorted(raw['k'].unique()))
print("Problems:", sorted(raw['i'].unique()))
raw[['i','j','k','t','p','r','n']].head()


Rows: 2,340
Total numeric entries: 2,340,000
Models: ['gpt-4.1', 'gpt-4o', 'o3', 'o4-mini']
Prompts: ['C', 'M', 'N']
Problems: ['APHCE', 'DDI', 'GEA', 'METC', 'PPI', 'PRESC']


,i,j,k,t,p,r,n
0,APHCE,gpt-4o,C,0.0,1.0,1,1000
1,APHCE,gpt-4o,C,0.0,1.0,2,1000
2,APHCE,gpt-4o,C,0.0,1.0,3,1000
3,APHCE,gpt-4o,C,0.0,1.0,4,1000
4,APHCE,gpt-4o,C,0.0,1.0,5,1000



## 2. Compute tables


In [ ]:
df_manuscript = tr.add_row_scores(raw, degenerate_policy="zero")
main = tr.write_all("manuscript_mode", df_manuscript, n_boot=N_BOOT)
df_strict = tr.add_row_scores(raw, degenerate_policy="floor")
strict = tr.write_all("strict_floor_mode", df_strict, n_boot=N_BOOT)
print("Saved CSV outputs to", RESULTS_DIR.resolve())


NameError: name 'tr' is not defined


## 3. Pooled outlier/nAURC table


In [4]:

pooled = main['table_pooled_outlier_nAURC'].copy()
show = pooled[['j','k','n','p_out_pct','p_out_pct_lo','p_out_pct_hi','nAURC','nAURC_lo','nAURC_hi']].copy()
for c in ['p_out_pct','p_out_pct_lo','p_out_pct_hi','nAURC','nAURC_lo','nAURC_hi']:
    show[c] = show[c].round(6)
show.sort_values(['j','k'])


,j,k,n,p_out_pct,p_out_pct_lo,p_out_pct_hi,nAURC,nAURC_lo,nAURC_hi
9,gpt-4.1,C,360000,5.603889,5.529229,5.679496,4.869686,3.265849,7.202513
10,gpt-4.1,M,360000,4.686111,4.617555,4.755634,3.908800,2.296133,6.577855
11,gpt-4.1,N,360000,2.350278,2.301296,2.400277,1.497322,0.596888,3.705471
0,gpt-4o,C,360000,2.600000,2.548519,2.652492,0.558047,0.080966,3.740992
1,gpt-4o,M,360000,0.187500,0.173890,0.202173,0.154856,0.052996,0.451609
2,gpt-4o,N,360000,0.039167,0.033214,0.046185,0.039983,0.019710,0.081088
6,o3,C,30000,8.593333,8.281461,8.915809,8.214496,6.690369,10.048444
7,o3,M,30000,9.483333,9.156958,9.820084,8.574836,6.900058,10.609790
8,o3,N,30000,0.013333,0.005185,0.034282,0.000006,0.000000,0.000254
3,o4-mini,C,30000,9.240000,8.917497,9.572941,8.883261,7.319741,10.742044



## 4. Heavy-tail diagnostics


In [5]:

tail = main['table_tail_diagnostics'].copy()
tail_show = tail[['j','k','log10_1p_AE','log10_1p_MAXEX','log10_R9990','log10_1p_S99','Moors']].copy()
for c in tail_show.columns[2:]:
    tail_show[c] = tail_show[c].round(6)
tail_show.sort_values(['j','k'])


,j,k,log10_1p_AE,log10_1p_MAXEX,log10_R9990,log10_1p_S99,Moors
9,gpt-4.1,C,2.628697,4.770209,1.155242,1.550891,1.714521
10,gpt-4.1,M,2.266477,4.462824,1.081480,1.390004,1.297318
11,gpt-4.1,N,0.607631,1.617683,0.497158,0.778864,1.380694
0,gpt-4o,C,0.967564,2.512458,0.510656,0.848793,1.352079
1,gpt-4o,M,0.511862,1.278410,0.171243,0.458845,1.217212
2,gpt-4o,N,0.205896,0.430158,0.121543,0.416885,1.069948
6,o3,C,1.793714,3.560280,1.433769,1.995990,2.297796
7,o3,M,1.428821,2.609430,1.328909,1.862488,2.816934
8,o3,N,0.051019,0.075140,0.066577,0.385707,0.983867
3,o4-mini,C,2.053264,3.777364,1.543838,2.211720,3.013566


In [6]:

alpha = main['table_tail_alpha'].copy()
alpha_show = alpha[['j','k','alpha','alpha_lo','alpha_hi']].copy()
for c in ['alpha','alpha_lo','alpha_hi']:
    alpha_show[c] = alpha_show[c].round(6)
alpha_show.sort_values(['j','k'])


,j,k,alpha,alpha_lo,alpha_hi
9,gpt-4.1,C,1.000020,0.999981,1.000059
10,gpt-4.1,M,1.000363,0.999657,1.001069
11,gpt-4.1,N,1.708984,1.124935,2.293033
0,gpt-4o,C,1.180075,0.991449,1.368700
1,gpt-4o,M,2.023233,1.163133,2.883332
2,gpt-4o,N,4.822489,0.661590,8.983388
6,o3,C,1.012512,0.996523,1.028502
7,o3,M,1.040078,0.986241,1.093915
8,o3,N,18.586072,1.000000,53.054773
3,o4-mini,C,1.001380,0.999062,1.003698



## 5. Dom. and m.-analysis


In [7]:

dom = main['table_dominance_curves'].copy()
dom_show = dom.copy()
for c in ['area','area_lo','area_hi']:
    dom_show[c] = dom_show[c].round(6)
dom_show.sort_values(['j','pair'])


,j,pair,area,area_lo,area_hi
0,gpt-4.1,N-C,-0.093748,-0.116223,-0.071658
1,gpt-4.1,N-M,-0.070504,-0.092025,-0.051720
2,gpt-4o,N-C,-0.071424,-0.084470,-0.055633
3,gpt-4o,N-M,-0.006028,-0.009697,-0.002979
4,o3,N-C,-0.251908,-0.312004,-0.190387
5,o3,N-M,-0.264158,-0.316851,-0.205786
6,o4-mini,N-C,-0.255617,-0.330838,-0.181269
7,o4-mini,N-M,-0.252642,-0.313677,-0.185690


In [8]:

meta = main['table_meta_pout'].copy()
meta_show = meta.copy()
for c in ['pooled_diff_pp','ci_low','ci_high','tau2']:
    meta_show[c] = meta_show[c].round(6)
meta_show.sort_values(['j','contrast'])


,j,contrast,pooled_diff_pp,ci_low,ci_high,tau2
0,gpt-4.1,C-N,3.253611,0.810451,5.696771,9.322729
1,gpt-4.1,M-N,2.335833,-1.108531,5.780197,18.529222
2,gpt-4o,C-N,2.560833,1.793336,3.328330,0.920010
3,gpt-4o,M-N,0.148333,0.055098,0.241569,0.013577
4,o3,C-N,8.580000,6.611069,10.548931,6.054803
5,o3,M-N,9.470000,7.583282,11.356718,5.559723
6,o4-mini,C-N,8.603333,6.629889,10.576778,6.082594
7,o4-mini,M-N,8.553333,6.432812,10.673854,7.023025



## 6. Temp. and top-p const


In [9]:

temp_strict = strict['table_temperature_pout'].copy()
temp_zero = main['table_temperature_pout'].copy()

cols = ['j','k','diff_pp','ci_low_pp','ci_high_pp']
print("Strict-floor temperature contrasts (matches manuscript figure/text):")
display(temp_strict[cols].sort_values(['j','k']).round(6))
print("\nZero-degenerate/manuscript-mode temperature contrasts:")
display(temp_zero[cols].sort_values(['j','k']).round(6))


Strict-floor temperature contrasts (matches manuscript figure/text):


,j,k,diff_pp,ci_low_pp,ci_high_pp
3,gpt-4.1,C,2.374444,2.214881,2.534165
4,gpt-4.1,M,2.027222,1.871729,2.182863
5,gpt-4.1,N,3.128333,3.022360,3.235048
0,gpt-4o,C,0.832222,0.721696,0.942917
1,gpt-4o,M,0.380000,0.351329,0.410347
2,gpt-4o,N,0.078333,0.066240,0.092367



Zero-degenerate/manuscript-mode temperature contrasts:


,j,k,diff_pp,ci_low_pp,ci_high_pp
3,gpt-4.1,C,1.201111,1.050937,1.351384
4,gpt-4.1,M,1.248889,1.110929,1.386985
5,gpt-4.1,N,3.309444,3.211535,3.408340
0,gpt-4o,C,0.802222,0.698364,0.906272
1,gpt-4o,M,0.347222,0.319743,0.376359
2,gpt-4o,N,0.078333,0.066240,0.092367


In [10]:

topp = main['table_top_p_pout'].copy()
topp[['j','k','contrast','diff_pp','ci_low_pp','ci_high_pp']].sort_values(['j','k']).round(6)


,j,k,contrast,diff_pp,ci_low_pp,ci_high_pp
3,gpt-4.1,C,0.90-1.00,-0.320000,-0.502132,-0.137909
4,gpt-4.1,M,0.90-1.00,0.467500,0.300534,0.634545
5,gpt-4.1,N,0.90-1.00,0.363333,0.242334,0.484480
0,gpt-4o,C,0.90-1.00,0.408333,0.282771,0.534046
1,gpt-4o,M,0.90-1.00,-0.040833,-0.069696,-0.012329
2,gpt-4o,N,0.90-1.00,0.045833,0.030099,0.062868
